# Libraries Used

In [1]:
import evaluate
import pandas as pd
import transformers
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers.trainer_utils import EvalPrediction


W0909 18:47:16.139000 14984 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


## Setup...

In [2]:
import sys

is_colab = "google.colab" in sys.modules
if is_colab:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    OUT_DIR = "./drive/MyDrive/Colab Notebooks/ModernBert"
else:
    OUT_DIR = "./ModernBert"

# instantiate the model
NAME = "answerdotai/ModernBERT-base"

In [3]:
def get_model():
    return transformers.AutoModelForSequenceClassification.from_pretrained(
        NAME,
        dtype="auto",
        num_labels=2,
    )


# set up the dataset and dataset loading
tokenizer = transformers.AutoTokenizer.from_pretrained(NAME)
ds_raw = load_dataset("stanfordnlp/sst2")

# tokenize the dataset entries
ds = ds_raw.map(
    lambda item: tokenizer(
        item["sentence"],
        truncation=True,
        max_length=512,
    )
)

# batch collator
collator = transformers.DataCollatorWithPadding(tokenizer)


def get_trainer(model, type: str):
    training_args = transformers.TrainingArguments(
        output_dir=f"{OUT_DIR}/{type}",
        num_train_epochs=4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        bf16=True,
        learning_rate=2e-5,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=1000,
        load_best_model_at_end=True,
    )

    return transformers.Trainer(
        model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        compute_metrics=calc_accuracy,
        processing_class=tokenizer,
        data_collator=collator,
    )

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

## Regular FT

### Training

In [4]:
model = get_model()

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# set up training params
for param in model.base_model.parameters():
    param.requires_grad_(False)

# accuracy setup
accuracy = evaluate.load("accuracy")


def calc_accuracy(p: EvalPrediction) -> dict:
    return accuracy.compute(
        predictions=p.predictions.argmax(axis=-1),  # type: ignore
        references=p.label_ids,
    )

trainer = get_trainer(model, "sft")

In [7]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
1000,4.613239,0.529529,0.768349
2000,4.126254,0.487601,0.779817
3000,3.908635,0.451164,0.802752
4000,3.785753,0.449284,0.791284
5000,3.691608,0.440595,0.797018
6000,3.619054,0.426529,0.802752
7000,3.547132,0.423267,0.813073
8000,3.523846,0.418352,0.805046
9000,3.584183,0.416245,0.816514
10000,3.499482,0.415085,0.816514


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16840, training_loss=3.6894643461902463, metrics={'train_runtime': 4814.7816, 'train_samples_per_second': 55.952, 'train_steps_per_second': 3.498, 'total_flos': 3511377825746616.0, 'train_loss': 3.6894643461902463, 'epoch': 4.0})

### Results

In [8]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
3.500054,0.410299,16840,0.811927


{'eval_loss': 0.4102989435195923, 'eval_accuracy': 0.8119266055045872}

## LoRA

Let's see how many params the traditioal model had...

In [9]:
trainer.get_num_trainable_parameters()

592130

Now let's see what we can get with LoRA

In [10]:
lora_config = LoraConfig(
    r=8,
    task_type=TaskType.SEQ_CLS,
    target_modules=['Wqkv', 'classifier', 'head.dense']
)
lora_model = get_peft_model(get_model(), lora_config)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
lora_model.get_nb_trainable_parameters()[0]

554498

So we have 50000 fewer parameters - about in the same ballpark.

Now let's train...

In [12]:
lora_trainer = get_trainer(lora_model, 'lora')

In [13]:
lora_trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Accuracy
1000,4.998755,0.590522,0.697248
2000,3.414246,0.401517,0.818807
3000,2.567927,0.294973,0.884174
4000,2.172429,0.265127,0.899083
5000,2.083864,0.253141,0.903670
6000,1.902052,0.253809,0.909404
7000,1.867612,0.236811,0.918578
8000,1.885964,0.227105,0.915138
9000,1.771654,0.233338,0.922018
10000,1.772497,0.228288,0.922018


TrainOutput(global_step=16840, training_loss=2.1987698552727415, metrics={'train_runtime': 11942.4854, 'train_samples_per_second': 22.558, 'train_steps_per_second': 1.41, 'total_flos': 3528930888009072.0, 'train_loss': 2.1987698552727415, 'epoch': 4.0})